# Laya multilingual × AI4I 2020 — 버전 비교 (FP32)

**Kaggle**: Accelerator **GPU T4**, Internet **On** → **Save Version → Save & Run All (Commit)** 추천 (전체 40~60분 예상, 창 닫아도 됨).
데이터(AI4I 원본, splits.csv)는 노트북 안에 들어 있다.

| 버전 | 특징값 | 학습 행 | 보는 것 |
|---|---|---|---|
| A | 성민 v7 그대로 | 500 | 성민 결과 재현 (기준) |
| B | v7 + 기준선 거리 | 500 | 특징값 효과 |
| C | v7 + 기준선 거리 | 1,000 | 정상 사례 2배 |
| D | v7 + 기준선 거리 | 2,000 | 정상 사례 4배 |

- 학습 설정은 성민 v7과 같다: 16에폭, micro 4, accum 4, rlcd, seed 0.
- 버전 선택은 **validation 80개**로만 한다. **test 80개는 고른 버전 하나만 마지막에 한 번** 본다 (`RUN_TEST`).
- AI4I에는 splits.csv 밖에 남은 깨끗한 고장 사례가 거의 없어서(열 방출 5개), 행을 늘리면 늘어나는 건 정상 사례다. 정상 사례는 고장 기준선 근처에서 고르게 뽑는다.
- 기준선 거리 특징값은 UCI 데이터 설명의 고장 공식(열 방출: 온도차 < 8.6K & 회전 < 1380rpm / 동력: < 3500W 또는 > 9000W / 과부하: 마모×토크 > 11,000·12,000·13,000)에서 나왔다. 보고서에 명시할 것.

In [1]:
import torch
assert torch.cuda.is_available(), "GPU가 안 잡혔어요. Accelerator를 GPU T4로 바꾸고 다시 실행하세요."
print(torch.cuda.get_device_name(0))

Tesla T4


In [2]:
!pip install -q "laya==0.4.31" scikit-learn tabulate

## 1. 설정

In [3]:
import os
os.environ["USE_TF"] = "0"
os.environ["LAYA_CUDA_AMP"] = "float32"   # 추론 FP32 고정

VERSIONS = [  # (이름, 특징값, 학습 행 수)
    ("A_v7_500", "v7", 500),
    ("B_margin_500", "margin", 500),
    ("C_margin_1000", "margin", 1000),
    ("D_margin_2000", "margin", 2000),
]
EPOCHS, MICRO, ACCUM, SEED = 16, 4, 4, 0
HARD_NORMAL_SHARE = 0.65
RUN_TEST = True   # True면 validation 최고 버전 하나만 test로 마지막에 한 번 채점

import base64, gzip, io, json, math, time, shutil
import pandas as pd, numpy as np
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
splits = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode("H4sIAFhW/GoC/61cu44mxRXOeYp5gJFc514nd+IHIEaD2GClNSB2gBSEA2dOvDJIyNoE+SIHlteQ2JLfZ2eQX4Hq/gftP4txn/3ORCPN31Vd5/ada/V///Wft3/5q8snV+8+enL54QefPvronXev3n/v8un1R1fXH//68umHTx5fvxVjzsuXL759+e3fLm7+9Nubv/754uWL57d//Oby9vmz2y8+v7j5x+9vv/7N/35k7fT4/bfEZDS3YFW6e/q1RXf/PD2myl54zCysslvwvLz9wz8vbv7+l9vvvr5Yx7l99vz1FT/5/bTYWUonZh+Fx8LbUli8GZc3v/vq5vk3r9Z+dvrH/f9f/OLi1S+nxbQtfvni3zffffb9sy9/7t2v/fyj6CwKFPqMkkhiFh6jyNdpXX9+htZXv/zIKGVY7Da0ooHTk7omYUNAkSwuEkxhWCr4Xh+jCwTqCisxhzN4ctMYJfWUhuoIPQBMGqz3PGfHaIbDlCunwFKdrBVIyEHZ1b1JhoJgKnffToZ7IyJWWLaZXnJl7rjyz+lS8RK2HDxKR5gErmaSCS92CmmKP5gqMQuLoBgXnhWMm5Sw3yFJXIXnwAWwYpAKSFiMivdWmgLTwWPCGiw+YQTyBribKA7uNnBY0Kl4pJKipZhy4ToeDc2Je1zlhh55LIdWyBySA1QZGumwxaUbqqoq68gwW5ZxVoSes4Em4Xi8ImwVb7pcRsKKNX0kynzfIl0Uo/kn1vAmGJ242Ckzmh423RVmWgfjzL0R2EUFBVhxn93IFY25cjqdxp18FI54pnUzLrKFkaBHHoEunSoTXJoDT6CVqaHmwXgQYToIJFiScSDVEXjEuuzaakFn4LmZO6oJrstwCsYp2UjrWL1RUzDcFVkIGn9ILdVStkZUrIxzVcJxTF7Q4U3Qm2LdajQ5bpQy4GL0qugTHlVFrbvAUko7nDpOzxtpxxiCM39lPA0f0IBTltFIYTMCrpYLTrGIoY6LV+ekWzPkxCteutIcXNJj4tpN1q6Ea2ilKKxwL2NVlKjRHLRutrIYjL+fBHaOaksrYWtgqXhWgvs0q6LS0Fma1oC2KaXqaYkBInBWoUNrdWI0I8zJeONRsyEemnBFyTpQJl0cXrohpbqyWyPbILywbBP1jivaqQVFcMyW0Yh1iKiS5PiW54G6pTQanU+djdpzNwhPvFoXW50QzK8i8cRpJV1aKillpfJkWkJtJx+4R9JGVxTPVlhRjU6LdnQyG1GCZ1JjUinx5Ma9UXRfKqcl326l6oIY3rGn/tBerrpqdwuF5yXmnKXaFAk1alP9aSTlzkxGY1hHqFZZp0aNKla9t/KOURrA8WRUG4gcjTh9DnzqkW0yjKG18c8YjUmCoIrzSi6NVuY+vnOcwBhbbUQCj4rWiAmc/kxD3Z474eBv1B7wioCDjNVTLjW9NfE0SEqIHFTSIqZZgi9r4Cs++KCbJuD1da9NALdrXGvEAq9eUaODsLpi+GJvdLuZBe9ergQZ70WJ2sTH/1LwmHSNeZfaCcl4rkA5SjP6XBuo4kZpdOFHrXui3RwlqFQjXjkBamUyG3mMT2mXiIPxbGbzRnjXI/Coa12hQPzwJ1dPHr93df34gyUyUfx2xdk+a8zzzUV/tn6lotGjZA7AiZ2tZ8smL3UAYfv5esn/H5qc8zv2Cay+3JAGwtl62Tu4b6z75zuMQT2+Rx4EUudvs4mY29kOZrMsJZUxe9yZzGXajKPOh7270jnZGlKi3g7k1FG9Vd2MKrlJYQ9hLiti5R7RMYibCrhXUooKaNY0T3VuUrw6uN6RMwU0KXtPVbtSY69DzKoFxYPoGkGduPMdlk9r0q1pPRudVuXbtClNByw5evTmlBYmrWHyrNLrp4vCbT1Z1/VaMqJtarFlX5LSsvA4mv48l/Gkh7GuzNmkelIZE5S6KKp7CtbZwTzL/pJtdAQ6Bbq5dU6uahO61k1MbZ6BuAknawjcm2ewLoTb1LKWukgzpCNgxvoefrL1GJZHtzzu+8e2VR5UMM+Ngg7u7d5LJTW7UWI206o1/OwPgrSzmyLGacgRT+09+CEoWdFMthL8Uc5X3HsUp856Kqhli1nDkK0ii7KNri4ENNr86On1pkeNT4lsO8hSpTevcO7vDlf03OtKz8DeqiroS9dghGMvXeNOBy2209Fi9ISxpg0FZMvEbmlva3lL71CextG3mXZxbyPHQEl4X5tjFHhP2F3xXbrjaDB5p5Tl+CFZcSJ6jFWXgEmIMNCKeT/xEV1rstqaqu0Dpw7LbE4nX1+2qKgPNjq+rw2sYbwr90Cm3U+a5jBPVHFZcHIBCuc2edvTF8KtYV2vg6XphPTVd4A4/IrNCWxgp6k2cXBJ5OMhd57FC8Dn6T2B05bWQSecCluCGCfIlhEFx6fbB1968dnm2FDso2GVMzbUShgOO+RwmGzHfiT/PikUQZn3HaYrzJM1mEPwewf2GYxd2gMbcbmTI+zD+PBm7F3cDmOyMKgCq3QCs4T5oLh456awr1eccEsrjDsqd50CCRgkaBAc/TN5NN28CazwoYGmc95M1NIddaeFEGE5a1DfnQrBGW9WcfhUDgLZq2Gwqf8Ah2MkpZhYAAA="))))
ai4i = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode("..."))), encoding="utf-8-sig")
print(ai4i.shape, splits.split.value_counts().to_dict())

(10000, 14) {'train': 241, 'validation': 80, 'test': 80}


## 2. 라벨, 특징값, 학습 데이터 구성

In [4]:
FAULT = {"TWF": "공구 마모 고장", "HDF": "열 방출 장애", "PWF": "동력 고장", "OSF": "과부하 고장"}
LIMIT = {"L": 11000, "M": 12000, "H": 13000}

def clean_label(r):   # 성민 v7과 같은 기준: 랜덤 고장·복수 고장 제외
    if r["RNF"] != 0: return None
    act = [k for k in FAULT if r[k] == 1]
    if len(act) == 1 and r["Machine failure"] == 1: return FAULT[act[0]]
    if not act and r["Machine failure"] == 0: return "정상"
    return None

ai4i["label_clean"] = ai4i.apply(clean_label, axis=1)
merged = splits.merge(ai4i, on="UDI", how="left", validate="one_to_one")
assert (merged["label"] == merged["label_clean"]).all(), "splits.csv 라벨과 원본 플래그가 다름"

def feats(r, features):
    air, proc = float(r["Air temperature [K]"]), float(r["Process temperature [K]"])
    rpm, tq, wear = int(r["Rotational speed [rpm]"]), float(r["Torque [Nm]"]), int(r["Tool wear [min]"])
    gap, power, wt, lim = proc - air, tq * rpm * 2 * math.pi / 60, wear * tq, LIMIT[r["Type"]]
    s = {"product_type": r["Type"], "air_temperature_K": air, "process_temperature_K": proc,
         "rotational_speed_rpm": rpm, "torque_Nm": tq, "tool_wear_min": wear,
         "temperature_gap_K": round(gap, 2), "power_W": round(power, 2), "wear_torque": round(wt, 2)}
    if features == "margin":
        s.update({
            "overstrain_limit_minNm": lim,
            "gap_minus_8.6K": round(gap - 8.6, 2),               # 음수면 열 방출 조건 1 충족
            "rpm_minus_1380": rpm - 1380,                         # 음수면 열 방출 조건 2 충족
            "power_minus_3500W": round(power - 3500, 1),          # 음수면 동력 부족
            "9000W_minus_power": round(9000 - power, 1),          # 음수면 동력 과다
            "overstrain_limit_minus_wear_torque": round(lim - wt, 1),  # 음수면 과부하
            "tool_wear_minus_200min": wear - 200,                 # 0~40이면 공구 교체·고장 구간
        })
    return s

QUESTIONS = {"diagnosis": {"type": "choice",
    "instructions": "밀링 공정 센서값과 계산값을 보고 설비 상태를 분류하세요.",
    "criteria": {
        "정상": "모든 고장 조건에서 벗어나 정상 작동",
        "열 방출 장애": "공기·공정 온도차가 작고 회전이 느려 열이 빠지지 않음",
        "동력 고장": "토크×회전속도로 계산한 동력이 너무 낮거나 너무 높음",
        "과부하 고장": "공구 마모×토크가 제품 유형별 한계를 넘음",
        "공구 마모 고장": "공구 마모 시간이 교체·고장 구간(약 200~240분)에 도달"}}}
LABELS = list(QUESTIONS["diagnosis"]["criteria"])

def boundary_dist(r):   # 정상 사례가 각 고장 기준선에 얼마나 가까운가 (작을수록 까다로움)
    gap = r["Process temperature [K]"] - r["Air temperature [K]"]
    rpm, tq, wear = r["Rotational speed [rpm]"], r["Torque [Nm]"], r["Tool wear [min]"]
    power = tq * rpm * 2 * math.pi / 60
    return pd.Series({"HDF": max(gap - 8.6, 0) / 1.0 + max(rpm - 1380, 0) / 150,
                      "PWF": min(abs(power - 3500), abs(power - 9000)) / 800,
                      "OSF": abs(LIMIT[r["Type"]] - wear * tq) / 1500,
                      "TWF": 0.0 if 200 <= wear <= 240 else 99.0})

held = set(merged.loc[merged.split != "train", "UDI"])
base = merged[merged.split == "train"].copy()
pool = ai4i[~ai4i.UDI.isin(set(merged.UDI)) & ai4i.label_clean.notna()].copy()
pool["label"] = pool["label_clean"]
extra_f = pool[pool.label != "정상"]
normals = pool[pool.label == "정상"].copy()
DIST = normals.apply(boundary_dist, axis=1)
in_window = (normals["Tool wear [min]"] >= 195) & (normals["Tool wear [min]"] <= 245)
print("남은 고장 사례:", extra_f.label.value_counts().to_dict(), "/ 남은 정상 사례:", len(normals))

def build_train(n_train):
    need = max(n_train - len(base) - len(extra_f), 0)
    n_hard = int(round(need * HARD_NORMAL_SHARE))
    quota = {"HDF": 0.3, "PWF": 0.3, "OSF": 0.3, "TWF": 0.1}   # 공구 마모 구간은 적게 (그 클래스를 해치지 않게)
    picked = []
    for kind, q in quota.items():
        cand = DIST.drop(index=picked)[kind]; cand = cand[cand < 99]
        if kind != "TWF": cand = cand[~in_window.loc[cand.index]]
        k = int(round(n_hard * q))
        picked += list(cand.sample(frac=1, random_state=SEED).nsmallest(k).index) if kind != "TWF" \
                  else list(cand.sample(n=min(k, len(cand)), random_state=SEED).index)
    hard = normals.loc[picked]
    rest_pool = normals.drop(index=picked); rest_pool = rest_pool[~in_window.loc[rest_pool.index]]
    rest = rest_pool.sample(n=need - len(hard), random_state=SEED)
    tr = pd.concat([base, extra_f, hard, rest], ignore_index=True)
    assert tr.UDI.is_unique and not (set(tr.UDI) & held)
    return tr

def build_train_v7(n_train):   # 성민 v7 방식: 남은 고장 우선 + 무작위 정상 (random_state=42)
    need = n_train - len(base)
    ef = extra_f.sample(n=min(len(extra_f), need), random_state=42)
    en = normals.sample(n=need - len(ef), random_state=42)
    tr = pd.concat([base, ef, en], ignore_index=True)
    assert tr.UDI.is_unique and not (set(tr.UDI) & held)
    return tr

def dump(df, path, features):
    with open(path, "w", encoding="utf-8") as f:
        for _, r in df.iterrows():
            f.write(json.dumps({"state": feats(r, features), "questions": QUESTIONS,
                                "expected": {"diagnosis": r["label"]}, "meta": {"UDI": int(r["UDI"])}},
                               ensure_ascii=False) + "\n")

남은 고장 사례: {'열 방출 장애': 5} / 남은 정상 사례: 9543


## 3. 평가 함수

In [5]:
import torch, laya
from sklearn.metrics import accuracy_score, f1_score, classification_report

def run(model, path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    for r in rows[:3]: model.predict(r["state"], QUESTIONS)
    out = []
    for r in rows:
        torch.cuda.synchronize(); t = time.perf_counter()
        a = model.predict(r["state"], QUESTIONS)["answers"]["diagnosis"]
        torch.cuda.synchronize()
        out.append({"UDI": r["meta"]["UDI"], "exp": r["expected"]["diagnosis"], "pred": a.get("choice"),
                    "ms": (time.perf_counter() - t) * 1000})
    return pd.DataFrame(out)

def score(df):
    return {"accuracy": round(accuracy_score(df.exp, df.pred), 4),
            "correct": f"{int((df.exp == df.pred).sum())}/{len(df)}",
            "macro_f1": round(f1_score(df.exp, df.pred, labels=LABELS, average="macro", zero_division=0), 4),
            "missed_failures": int(((df.exp != "정상") & (df.pred == "정상")).sum()),
            "tool_wear_recall": round(float((df[df.exp == "공구 마모 고장"].pred == "공구 마모 고장").mean()), 4),
            "valid_output_rate": round(float(df.pred.isin(LABELS).mean()), 4),
            "latency_ms_p50": round(float(df.ms.median()), 2)}

## 4. 버전별 파인튜닝 → FP32 저장 → validation 채점

In [6]:
import safetensors.torch as st
from laya.train import finetune, TrainConfig
_orig_save = st.save_file
def _save_fp32(tensors, filename, metadata=None):
    return _orig_save({k: (v.float() if v.is_floating_point() else v) for k, v in tensors.items()}, filename, metadata=metadata)
st.save_file = _save_fp32

results = {}
val_path = f"{WORK}/ai4i_validation_%s.jsonl"
for name, features, n_train in VERSIONS:
    print(f"\n========== {name} ({features}, {n_train}행) ==========")
    tr = build_train_v7(n_train) if features == "v7" else build_train(n_train)
    print("학습 라벨:", tr.label.value_counts().to_dict())
    dump(tr, f"{WORK}/ai4i_train_{name}.jsonl", features)
    if not os.path.exists(val_path % features):
        dump(merged[merged.split == "validation"], val_path % features, features)
        dump(merged[merged.split == "test"], f"{WORK}/ai4i_test_{features}.jsonl", features)
    ckpt = f"{WORK}/ckpt_{name}"
    cfg = TrainConfig(epochs=EPOCHS, micro_batch=MICRO, grad_accum=ACCUM, loss="rlcd",
                      shuffle_options=("choice",), seed=SEED, eval_data=val_path % features)
    t0 = time.time()
    summ = finetune(f"{WORK}/ai4i_train_{name}.jsonl", "multilingual", ckpt, cfg)
    train_s = round(time.time() - t0)
    model = laya.load(ckpt)
    val = run(model, val_path % features); val.to_csv(f"{WORK}/val_preds_{name}.csv", index=False, encoding="utf-8-sig")
    m = score(val)
    m.update({"features": features, "n_train": len(tr), "train_s": train_s,
              "final_epoch_loss": round(summ.get("epoch_loss", [None])[-1] or 0, 4),
              "calib_issues": summ.get("calibration", {}).get("choice", {}).get("issues", [])})
    results[name] = m
    print(json.dumps(m, ensure_ascii=False))
    print(pd.crosstab(val.exp, val.pred).to_markdown())
    del model; torch.cuda.empty_cache()
    json.dump(results, open(f"{WORK}/ai4i_sweep_validation.json", "w"), ensure_ascii=False, indent=1)


========== A_v7_500 (v7, 500행) ==========
학습 라벨: {'정상': 314, '열 방출 장애': 66, '동력 고장': 48, '과부하 고장': 46, '공구 마모 고장': 26}


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

eval items 80 from /kaggle/working/ai4i_validation_v7.jsonl (skipped {}, 0 overlapping training/calibration items)
train items 450, calibration items 50, eval items 80 (/kaggle/working/ai4i_validation_v7.jsonl), skipped {}, device cuda:0
optimizer budget: 450 train item(s), effective batch up to 16 (micro_batch 4 x grad_accum 4), 464 optimizer updates over 16 epoch(s)


/usr/local/lib/python3.13/dist-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/16 step 100 loss 1.1378
epoch 1/16 mean loss 1.3338
epoch 2/16 step 100 loss 0.0246
epoch 2/16 mean loss 1.0708
epoch 3/16 step 100 loss 0.1265
epoch 3/16 mean loss 1.0546
epoch 4/16 step 100 loss 1.2940
epoch 4/16 mean loss 1.0349
epoch 5/16 step 100 loss 0.2089
epoch 5/16 mean loss 0.7998
epoch 6/16 step 100 loss 0.4003
epoch 6/16 mean loss 1.2479
epoch 7/16 step 100 loss -0.0794
epoch 7/16 mean loss 0.8047
epoch 8/16 step 100 loss 0.4933
epoch 8/16 mean loss 0.8543
epoch 9/16 step 100 loss 0.3163
epoch 9/16 mean loss 0.8265
epoch 10/16 step 100 loss 0.8725
epoch 10/16 mean loss 0.6163
epoch 11/16 step 100 loss 0.2889
epoch 11/16 mean loss 0.5817
epoch 12/16 step 100 loss 1.3438
epoch 12/16 mean loss 0.4515
epoch 13/16 step 100 loss 0.3939
epoch 13/16 mean loss 0.3395
epoch 14/16 step 100 loss 0.0070
epoch 14/16 mean loss 0.2552
epoch 15/16 step 100 loss 4.2998
epoch 15/16 mean loss 0.2406
epoch 16/16 step 100 loss -0.2598
epoch 16/16 mean loss 0.2441

=== Evaluation (before 

... (11 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

eval items 80 from /kaggle/working/ai4i_validation_margin.jsonl (skipped {}, 0 overlapping training/calibration items)
train items 1800, calibration items 200, eval items 80 (/kaggle/working/ai4i_validation_margin.jsonl), skipped {}, device cuda:0
optimizer budget: 1800 train item(s), effective batch up to 16 (micro_batch 4 x grad_accum 4), 1808 optimizer updates over 16 epoch(s)


/usr/local/lib/python3.13/dist-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/16 step 100 loss -0.0243
epoch 1/16 step 200 loss 4.3980
epoch 1/16 step 300 loss 4.2268
epoch 1/16 step 400 loss -0.0569
epoch 1/16 mean loss 1.6326
epoch 2/16 step 100 loss -0.0043
epoch 2/16 step 200 loss -0.0182
epoch 2/16 step 300 loss -0.0619
epoch 2/16 step 400 loss 3.8932
epoch 2/16 mean loss 1.6231
epoch 3/16 step 100 loss -0.0829
epoch 3/16 step 200 loss 4.3817
epoch 3/16 step 300 loss -0.0144
epoch 3/16 step 400 loss 4.4373
epoch 3/16 mean loss 1.6220
epoch 4/16 step 100 loss -0.0558
epoch 4/16 step 200 loss 4.3940
epoch 4/16 step 300 loss 4.4399
epoch 4/16 step 400 loss -0.0123
epoch 4/16 mean loss 1.6398
epoch 5/16 step 100 loss -0.0740
epoch 5/16 step 200 loss 0.0395
epoch 5/16 step 300 loss -0.0191
epoch 5/16 step 400 loss 4.0050
epoch 5/16 mean loss 1.6218
epoch 6/16 step 100 loss -0.0000
epoch 6/16 step 200 loss -0.0350
epoch 6/16 step 300 loss 3.4142
epoch 6/16 step 400 loss -0.0000
epoch 6/16 mean loss 1.1037
epoch 7/16 step 100 loss 0.0000
epoch 7/16 step 20

## 5. validation 비교표

In [7]:
tbl = pd.DataFrame(results).T[["features", "n_train", "accuracy", "correct", "macro_f1", "missed_failures",
                                "tool_wear_recall", "valid_output_rate", "latency_ms_p50", "train_s", "final_epoch_loss"]]
print(tbl.to_markdown())
tbl.to_csv(f"{WORK}/ai4i_sweep_validation.csv", encoding="utf-8-sig")
# 고르는 기준: 정확도 → Macro-F1 → 놓친 고장 적은 순
best = sorted(results, key=lambda k: (-results[k]["accuracy"], -results[k]["macro_f1"], results[k]["missed_failures"]))[0]
print("\nvalidation 기준 최고 버전:", best)
print("참고: 24회 모델 검증 78.75%, 성민 v7 검증 87.5~91.25%")

|               | features   |   n_train |   accuracy | correct   |   macro_f1 |   missed_failures |   tool_wear_recall |   valid_output_rate |   latency_ms_p50 |   train_s |   final_epoch_loss |
|:--------------|:-----------|----------:|-----------:|:----------|-----------:|------------------:|-------------------:|--------------------:|-----------------:|----------:|-------------------:|
| A_v7_500      | v7         |       500 |     0.825  | 66/80     |     0.7962 |                 7 |              0.5   |                   1 |            33.6  |       474 |             0.2441 |
| B_margin_500  | margin     |       500 |     0.925  | 74/80     |     0.8727 |                 5 |              0.375 |                   1 |            48.5  |       534 |             0.3913 |
| C_margin_1000 | margin     |      1000 |     0.8625 | 69/80     |     0.7719 |                 7 |              0.125 |                   1 |            48.35 |       981 |             0.2232 |
| D_margin_2000 | ma

## 6. test — 고른 버전 하나만, 한 번만

In [8]:
final = {"selected_by_validation": best, "validation": results[best]}
if RUN_TEST:
    feat = results[best]["features"]
    model = laya.load(f"{WORK}/ckpt_{best}")
    test = run(model, f"{WORK}/ai4i_test_{feat}.jsonl"); test.to_csv(f"{WORK}/test_preds_{best}.csv", index=False, encoding="utf-8-sig")
    final["test"] = score(test)
    print(json.dumps(final["test"], ensure_ascii=False, indent=1))
    print(classification_report(test.exp, test.pred, labels=LABELS, zero_division=0))
    print(pd.crosstab(test.exp, test.pred).to_markdown())
json.dump(final, open(f"{WORK}/ai4i_final.json", "w"), ensure_ascii=False, indent=1)

{
 "accuracy": 0.9625,
 "correct": "77/80",
 "macro_f1": 0.9399,
 "missed_failures": 3,
 "tool_wear_recall": 0.625,
 "valid_output_rate": 1.0,
 "latency_ms_p50": 48.9
}
              precision    recall  f1-score   support

          정상       0.87      1.00      0.93        20
     열 방출 장애       1.00      1.00      1.00        20
       동력 고장       1.00      1.00      1.00        16
      과부하 고장       1.00      1.00      1.00        16
    공구 마모 고장       1.00      0.62      0.77         8

    accuracy                           0.96        80
   macro avg       0.97      0.93      0.94        80
weighted avg       0.97      0.96      0.96        80

| exp            |   공구 마모 고장 |   과부하 고장 |   동력 고장 |   열 방출 장애 |   정상 |
|:---------------|-----------------:|--------------:|------------:|---------------:|-------:|
| 공구 마모 고장 |                5 |             0 |           0 |              0 |      3 |
| 과부하 고장    |                0 |            16 |           0 |              0 |      0 |

## 7. 저장 (최고 버전만 zip, 나머지 체크포인트는 디스크 절약을 위해 삭제)

In [9]:
from safetensors import safe_open
ck = f"{WORK}/ckpt_{best}"
with safe_open(f"{ck}/model.safetensors", "pt") as f:
    print("가중치 dtype:", {str(f.get_slice(k).get_dtype()) for k in f.keys()},
          f"/ {os.path.getsize(f'{ck}/model.safetensors') / 2**20:.0f} MB")
json.dump({"questions": QUESTIONS, "version": best, "features": results[best]["features"], "n_train": results[best]["n_train"],
           "epochs": EPOCHS, "micro_batch": MICRO, "grad_accum": ACCUM, "seed": SEED, "laya": "0.4.31"},
          open(f"{ck}/run_config.json", "w"), ensure_ascii=False, indent=1)
shutil.make_archive(f"{WORK}/laya-ai4i-fp32-{best}", "zip", ck)
for name, *_ in VERSIONS:
    if name != best: shutil.rmtree(f"{WORK}/ckpt_{name}", ignore_errors=True)
print("Output에서 받기: ai4i_sweep_validation.csv, ai4i_final.json, laya-ai4i-fp32-" + best + ".zip")

가중치 dtype: {'F32'} / 1228 MB
Output에서 받기: ai4i_sweep_validation.csv, ai4i_final.json, laya-ai4i-fp32-B_margin_500.zip
